# GeoMarket-VLC · Pipeline de Unión de Datos
**Asignatura:** Evaluación, Despliegue y Monitorización de Modelos (EDM)  
**Objetivo de este notebook:** Cargar todos los datasets disponibles, limpiarlos, y cruzarlos a nivel de **barrio** para obtener un único GeoDataFrame enriquecido listo para modelado y visualización.

> ⚠️ Ajusta la variable `DATA_DIR` de la celda siguiente para que apunte a la carpeta donde tienes los archivos.

## 0. Instalación de dependencias

In [1]:
# Ejecuta solo si no tienes estas librerías instaladas
# !pip install geopandas folium mapclassify shapely pyproj

## 1. Imports y configuración de rutas

In [2]:
import pandas as pd
import geopandas as gpd
import json
import unicodedata
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── AJUSTA ESTA RUTA ──────────────────────────────────────────────────────────
DATA_DIR = Path(".")   # carpeta donde están todos tus archivos
# ─────────────────────────────────────────────────────────────────────────────

# Función de normalización usada en todos los joins por nombre de texto
def normalizar(s):
    return (unicodedata.normalize('NFD', str(s).strip().lower())
            .encode('ascii', 'ignore').decode('utf-8'))

print("Archivos encontrados:")
for f in sorted(DATA_DIR.iterdir()):
    print(" ", f.name)

c:\Users\oscar\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Archivos encontrados:
  .ipynb_checkpoints
  airbnb_listings_limpio.csv.gz
  aparcamientos_barrios.csv
  aparcamientos_distritos.csv
  app.py
  app_2.py
  barris.json
  cache
  datos_comercios.ipynb
  demografia_distritos.csv
  emt_paradas.json
  equipament_municipal.json
  fgv_bocas.json
  geomarket_vlc_modelado.ipynb
  geomarket_vlc_pipeline.ipynb
  geomarket_vlc_pipeline_2.ipynb
  geomarket_vlc_pipeline_corregido.ipynb
  geomarket_vlc_pipeline_v2.ipynb
  geomarket_vlc_pipeline_v3.ipynb
  geomarket_vlc_pipeline_v4.ipynb
  geomarket_vlc_pipeline_v5.ipynb
  indice_turistico_por_barrio.csv
  locales_valencia.json
  locales_valencia_equil.json
  models
  output
  paradas_tren.json
  preprocesar_aparcamientos.py
  preu_habitatge-val_USELESS.csv
  prototipo_streamlit.ipynb
  requirements.txt
  secciones_censales.xml
  shap_explainer copy.pkl
  vulnerabilidad-por-barrios.csv


## 2. Carga de la geometría base: límites de barrios de Valencia

Este es el **esqueleto** del mapa. Todos los demás datasets se unirán a este GeoDataFrame usando el código o nombre de barrio como clave.

In [3]:
# FIX: gpd.read_file lee GeoJSON estándar directamente, sin parsear rings/attributes a mano
gdf_barrios = gpd.read_file(DATA_DIR / "barris.json")

if gdf_barrios.crs is None or gdf_barrios.crs.to_epsg() == 25830:
    gdf_barrios = gdf_barrios.set_crs('EPSG:25830').to_crs('EPSG:4326')
elif gdf_barrios.crs.to_epsg() != 4326:
    gdf_barrios = gdf_barrios.to_crs('EPSG:4326')

# Clave de join normalizada por nombre de barrio
gdf_barrios['barrio_key'] = gdf_barrios['nombre'].apply(normalizar)

print("CRS:", gdf_barrios.crs)
print("Columnas:", gdf_barrios.columns.tolist())
print("Nº barrios:", len(gdf_barrios))
gdf_barrios.head()
# FIX: el GeoJSON fuente tiene coddistbar='175' asignado a dos barrios distintos
# (RAFALELL-VISTABELLA y MAHUELLA-TAULADELLA) — error del dato original del Ayuntamiento.
# Convertimos coddistbar a entero y renombramos el duplicado para que ambos sean únicos.
gdf_barrios['coddistbar'] = pd.to_numeric(gdf_barrios['coddistbar'], errors='coerce').astype('Int64')
dup_mask = gdf_barrios.duplicated(subset=['coddistbar'], keep='first')
if dup_mask.sum() > 0:
    print(f"⚠ Duplicados detectados: {gdf_barrios[dup_mask][['coddistbar','nombre']].to_string()}")
    # Al duplicado le asignamos el siguiente código libre del mismo distrito
    max_d17 = gdf_barrios[gdf_barrios['coddistrit'].astype(str) == '17']['coddistbar'].max()
    gdf_barrios.loc[dup_mask, 'coddistbar'] = max_d17 + 1
    print(f"  → Reasignado a coddistbar={max_d17 + 1}")
print(f"Total barrios únicos: {gdf_barrios['coddistbar'].nunique()}")


CRS: EPSG:4326
Columnas: ['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry', 'barrio_key']
Nº barrios: 88
⚠ Duplicados detectados:     coddistbar               nombre
86         175  MAHUELLA-TAULADELLA
  → Reasignado a coddistbar=178
Total barrios únicos: 88


## 3. Dataset: Vulnerabilidad por barrios
Índices socioeconómicos y demográficos 2021. Es la fuente principal para el **Perfil B (Planificador Urbano)**.

In [4]:
df_vuln = pd.read_csv(DATA_DIR / "vulnerabilidad-por-barrios.csv", encoding='utf-8-sig', sep=None, engine='python')

print("Shape:", df_vuln.shape)
print("Columnas:", df_vuln.columns.tolist())
df_vuln.head(3)

Shape: (70, 15)
Columnas: ['Geo Point', 'Geo Shape', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


,Geo Point,Geo Shape,Nombre,Codbar,Distrito,Ind_Equip,Vul_Equip,Ind_Dem,Vul_Dem,Ind_Econom,Vul_Econom,Ind_Global,Vul_Global,Shape_Leng,Shape_Area
0,"39.489296787269716, -0.37938901146529563","{""coordinates"": [[[-0.382164770971817, 39.4902...",TORMOS,54,LA SAÏDIA,2.6,Vulnerabilidad Baja,2.9,Vulnerabilidad Baja,2.2,Vulnerabilidad Media,2.6,Vulnerabilidad Baja,2635.244586,279947.38785
1,"39.48407374709207, -0.374776457256648","{""coordinates"": [[[-0.369116459788766, 39.4873...",MORVEDRE,52,LA SAÏDIA,2.0,Vulnerabilidad Baja,2.4,Vulnerabilidad Alta,2.4,Vulnerabilidad Baja,2.3,Vulnerabilidad Alta,3081.246636,428183.60405
2,"39.48318799634565, -0.38580008084266776","{""coordinates"": [[[-0.39011563889024, 39.48553...",LES TENDETES,42,CAMPANAR,2.2,Vulnerabilidad Baja,3.0,Vulnerabilidad Baja,2.9,Vulnerabilidad Baja,2.7,Vulnerabilidad Baja,2527.930072,258207.64220


In [5]:
COL_BARRIO_VULN = 'Distrito'

df_vuln['barrio_key'] = df_vuln['Nombre'].apply(normalizar)
gdf_barrios['barrio_key'] = gdf_barrios['nombre'].apply(normalizar)

match = set(df_vuln['barrio_key']) & set(gdf_barrios['barrio_key'])
print(f"Coinciden por nombre: {len(match)} / {len(df_vuln)} vuln, {len(gdf_barrios)} geometría")

no_match_vuln = set(df_vuln['barrio_key']) - set(gdf_barrios['barrio_key'])
no_match_geo  = set(gdf_barrios['barrio_key']) - set(df_vuln['barrio_key'])
print("\nEn vulnerabilidad pero NO en geometría:", sorted(no_match_vuln))
print("\nEn geometría pero NO en vulnerabilidad:", sorted(no_match_geo))

# NOTA: los barrios en 'no_match_geo' son las pedanías de los distritos 17, 18 y 19
# (municipios agregados a Valencia que conservan su propia identidad administrativa).
# El dataset de vulnerabilidad municipal NO cubre estas zonas; es ausencia real de datos,
# NO un error de normalización de nombres.
print(f"\n{len(no_match_geo)} barrios sin datos de vulnerabilidad "
      f"(pedanías de distritos 17/18/19 — ausencia real de datos, no bug de normalización)")

# Corrección del desajuste conocido: 'mont-olivet' en vuln → 'montolivet' en geometría
df_vuln['barrio_key'] = df_vuln['barrio_key'].replace('mont-olivet', 'montolivet')

match = set(df_vuln['barrio_key']) & set(gdf_barrios['barrio_key'])
print(f"\nCoinciden tras corrección mont-olivet→montolivet: {len(match)} / {len(df_vuln)}")

gdf_barrios = gdf_barrios.merge(
    df_vuln.drop(columns=['Geo Point', 'Geo Shape'], errors='ignore'),
    on='barrio_key',
    how='left'
)

print("\nShape tras merge:", gdf_barrios.shape)
print("Columnas:", gdf_barrios.columns.tolist())
gdf_barrios[['nombre', 'Ind_Global', 'Vul_Global']].head(5)


Coinciden por nombre: 69 / 70 vuln, 88 geometría

En vulnerabilidad pero NO en geometría: ['mont-olivet']

En geometría pero NO en vulnerabilidad: ['benifaraig', 'beniferri', 'benimamet', 'borboto', 'carpesa', "castellar-l'oliveral", "el forn d'alcedo", 'el palmar', 'el perellonet', 'el saler', 'faitanar', 'la torre', 'les cases de barcena', 'mahuella-tauladella', 'massarrojos', 'montolivet', 'pinedo', 'poble nou', 'rafalell-vistabella']

19 barrios sin datos de vulnerabilidad (pedanías de distritos 17/18/19 — ausencia real de datos, no bug de normalización)

Coinciden tras corrección mont-olivet→montolivet: 70 / 70

Shape tras merge: (88, 21)
Columnas: ['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry', 'barrio_key', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


,nombre,Ind_Global,Vul_Global
0,BENIFARAIG,NaN,NaN
1,BENICALAP,3.4,Vulnerabilidad Baja
2,TORREFIEL,3.0,Vulnerabilidad Baja
3,TORMOS,2.6,Vulnerabilidad Baja
4,SANT ANTONI,2.6,Vulnerabilidad Baja


## 4. Dataset: Demografía por distritos

In [6]:
# FIX: carga limpia (evita el doble-prefix de Jupyter si la celda se ejecutó más de una vez)
df_demo_raw = pd.read_csv(DATA_DIR / "demografia_distritos.csv",
                          encoding='utf-8-sig', sep=None, engine='python')

# FIX: clave de distrito normalizada para el join posterior
df_demo_raw['districte_key'] = df_demo_raw['DISTRICTE'].apply(normalizar)

# FIX: pivot de formato largo (361 filas = 19 distritos x 19 grupos edad)
#      a formato ancho (19 filas = 1 por distrito) para evitar el many-to-many
df_demo_pivot = (
    df_demo_raw
    .pivot_table(index='districte_key',
                 columns='GRUP_EDAD',
                 values='HABITANTS_GRUP',
                 aggfunc='sum')
    .reset_index()
)

# Limpiar nombres de columna
df_demo_pivot.columns = (
    ['districte_key'] +
    ['demo_edad_' + str(c).strip().replace(' ', '_').replace('-', '_').replace('?', 'mas')
     for c in df_demo_pivot.columns[1:]]
)

# Features resumen útiles para el modelo
edad_cols = [c for c in df_demo_pivot.columns if c.startswith('demo_edad_')]
total_hab  = df_demo_pivot[edad_cols].sum(axis=1)

# FIX: los nombres de columna usan triple guion ("0 - 4" → "0___4", "65 - 69" → "65___69")
# porque replace(' ','_') + replace('-','_') sobre " - " produce "___"
SUFIJOS_JOVEN = ['0___4', '05___09', '10___14', '15___19', '20___24']
SUFIJOS_MAYOR = ['65___69', '70___74', '75___79', '80___84', '85___89', '90+']
cols_joven = [c for c in edad_cols if any(c.endswith(s) for s in SUFIJOS_JOVEN)]
cols_mayor = [c for c in edad_cols if any(c.endswith(s) for s in SUFIJOS_MAYOR)]

df_demo_pivot['demo_pct_joven']       = (df_demo_pivot[cols_joven].sum(axis=1) / total_hab * 100).round(2)
df_demo_pivot['demo_pct_mayor']       = (df_demo_pivot[cols_mayor].sum(axis=1) / total_hab * 100).round(2)
df_demo_pivot['demo_total_distrito']  = total_hab

print(f"Demografía pivotada: {df_demo_pivot.shape}  →  1 fila por distrito, sin explosión many-to-many")
df_demo_pivot[['districte_key', 'demo_pct_joven', 'demo_pct_mayor', 'demo_total_distrito']].head()

Demografía pivotada: (19, 23)  →  1 fila por distrito, sin explosión many-to-many


,districte_key,demo_pct_joven,demo_pct_mayor,demo_total_distrito
0,algiros,21.11,24.09,36711.0
1,benicalap,25.13,17.95,46699.0
2,benimaclet,21.55,21.72,28718.0
3,camins al grau,25.22,18.09,65451.0
4,campanar,24.53,20.36,38338.0


## 5. Dataset: Locales de Valencia (actividades comerciales)
Es el dataset más importante: contiene la localización puntual de cada local/negocio.

In [7]:
with open(DATA_DIR / "locales_valencia.json", encoding='utf-8') as f:
    locales_raw = json.load(f)

if isinstance(locales_raw, dict) and 'features' in locales_raw:
    gdf_locales = gpd.GeoDataFrame.from_features(locales_raw['features'], crs='EPSG:4326')
else:
    df_locales = pd.DataFrame(locales_raw if isinstance(locales_raw, list) else locales_raw.get('results', []))
    if 'geo_point_2d' in df_locales.columns:
        df_locales['lat'] = df_locales['geo_point_2d'].apply(lambda x: x['lat'] if isinstance(x, dict) else None)
        df_locales['lon'] = df_locales['geo_point_2d'].apply(lambda x: x['lon'] if isinstance(x, dict) else None)
    gdf_locales = gpd.GeoDataFrame(
        df_locales,
        geometry=gpd.points_from_xy(df_locales['lon'], df_locales['lat']),
        crs='EPSG:4326'
    )

print(f"Locales cargados: {len(gdf_locales)}")
print("Columnas:", gdf_locales.columns.tolist())
gdf_locales.head(3)

Locales cargados: 18856
Columnas: ['geometry', 'id', 'lat', 'lon', 'nombre', 'tipo']


,geometry,id,lat,lon,nombre,tipo
0,POINT (-0.38186 39.47101),30999260,39.471011,-0.381860,Ni pa ti ni pa mi,cafe
1,POINT (-0.38141 39.47090),30999262,39.470905,-0.381409,Poquito a poco,cafe
2,POINT (-0.38040 39.47118),31023735,39.471183,-0.380398,Cervecería Velluters,cafe


In [8]:
print(gdf_barrios.columns.tolist())

['objectid', 'codbarrio', 'nombre', 'coddistbar', 'coddistrit', 'gis.gis.BARRIOS.area', 'geometry', 'barrio_key', 'Nombre', 'Codbar', 'Distrito', 'Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem', 'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global', 'Shape_Leng', 'Shape_Area']


In [9]:
# Spatial join: asignar a cada local el barrio en el que está geométricamente
# Spatial join: asignar a cada local el barrio en que está
gdf_locales_con_barrio = gpd.sjoin(
    gdf_locales,
    gdf_barrios[['coddistbar', 'codbarrio', 'nombre', 'coddistrit', 'barrio_key', 'geometry']],
    how='left',
    predicate='within'
)

sin_barrio = gdf_locales_con_barrio['coddistbar'].isna().sum()
print(f"Locales sin barrio asignado (fuera del límite municipal): {sin_barrio}")
gdf_locales_con_barrio.head(3)

Locales sin barrio asignado (fuera del límite municipal): 5475


,geometry,id,lat,lon,nombre_left,tipo,index_right,coddistbar,codbarrio,nombre_right,coddistrit,barrio_key
0,POINT (-0.38186 39.47101),30999260,39.471011,-0.381860,Ni pa ti ni pa mi,cafe,25.0,14,4,EL PILAR,1,el pilar
1,POINT (-0.38141 39.47090),30999262,39.470905,-0.381409,Poquito a poco,cafe,25.0,14,4,EL PILAR,1,el pilar
2,POINT (-0.38040 39.47118),31023735,39.471183,-0.380398,Cervecería Velluters,cafe,25.0,14,4,EL PILAR,1,el pilar


In [10]:
df_locales_por_barrio = (
    gdf_locales_con_barrio
    .groupby('coddistbar')
    .size()
    .reset_index(name='n_locales_total')
)

COL_SECTOR = 'tipo'

if COL_SECTOR in gdf_locales_con_barrio.columns:
    df_locales_pivot = (
        gdf_locales_con_barrio
        .groupby(['coddistbar', COL_SECTOR])
        .size()
        .unstack(fill_value=0)
        .add_prefix('n_locales_')
        .reset_index()
    )
    df_locales_por_barrio = df_locales_por_barrio.merge(df_locales_pivot, on='coddistbar', how='left')

print("Shape agregado por barrio:", df_locales_por_barrio.shape)
df_locales_por_barrio.head(3)

Shape agregado por barrio: (84, 19)


,coddistbar,n_locales_total,n_locales_bakery,n_locales_bank,n_locales_bar,n_locales_cafe,n_locales_clothes,n_locales_convenience,n_locales_fast_food,n_locales_hairdresser,n_locales_health_care,n_locales_home_craft_repairs,n_locales_leisure_education,n_locales_otro,n_locales_pharmacy,n_locales_professional_services,n_locales_restaurant,n_locales_retail_commerce,n_locales_supermarket_food
0,11,245,4,0,18,18,7,7,5,1,0,5,4,99,2,0,51,22,2
1,12,150,2,2,3,6,6,1,3,1,2,5,3,86,1,2,22,4,1
2,13,337,4,0,18,11,3,7,1,1,1,4,6,209,2,0,57,9,4


## 6. Dataset: Paradas EMT (autobuses)

In [11]:
with open(DATA_DIR / "emt_paradas.json", encoding='utf-8') as f:
    emt_raw = json.load(f)
# El JSON de EMT es formato ESRI REST con "features":[{attributes, geometry}]
rows_emt = []
for feat in emt_raw['features']:
    attr = feat['attributes']
    geom = feat.get('geometry', {})
    attr['x'] = geom.get('x')
    attr['y'] = geom.get('y')
    rows_emt.append(attr)

df_emt = pd.DataFrame(rows_emt)
# Coordenadas en EPSG:25830 → convertir a WGS84
gdf_emt = gpd.GeoDataFrame(
    df_emt,
    geometry=gpd.points_from_xy(df_emt['x'], df_emt['y']),
    crs='EPSG:25830'
).to_crs('EPSG:4326')

print(f"Paradas EMT: {len(gdf_emt)}")


gdf_emt_barrio = gpd.sjoin(
    gdf_emt[['geometry']],
    gdf_barrios[['coddistbar', 'geometry']],  # ← codbarrio → coddistbar
    how='left', predicate='within'
)
df_emt_por_barrio = gdf_emt_barrio.groupby('coddistbar').size().reset_index(name='n_paradas_emt')

Paradas EMT: 1126


In [12]:

print(f"Barrios con paradas EMT: {len(df_emt_por_barrio)}")

Barrios con paradas EMT: 84


## 7. Dataset: Bocas de metro / FGV

In [13]:
# FIX: carga desde archivo local fgv_bocas.json (GeoJSON estándar)
gdf_tren = gpd.read_file(DATA_DIR / "fgv_bocas.json")

if gdf_tren.crs is None or gdf_tren.crs.to_epsg() != 4326:
    gdf_tren = gdf_tren.to_crs('EPSG:4326')

gdf_tren_barrio = gpd.sjoin(
    gdf_tren[['geometry']],
    gdf_barrios[['coddistbar', 'geometry']],  # ← codbarrio → coddistbar
    how='left', predicate='within'
)
df_tren_por_barrio = gdf_tren_barrio.groupby('coddistbar').size().reset_index(name='n_paradas_metro')  # ← igual



print(f"Bocas FGV cargadas: {len(gdf_tren)}")
print(f"Barrios con metro: {len(df_tren_por_barrio)}")



Bocas FGV cargadas: 187
Barrios con metro: 41


## 8. Dataset: Equipamiento municipal

In [14]:
with open(DATA_DIR / "equipament_municipal.json", encoding='utf-8') as f:
    equip_raw = json.load(f)

rows_equip = []
for feat in equip_raw['features']:
    attr = feat['attributes']
    geom = feat.get('geometry', {})
    attr['x'] = geom.get('x')
    attr['y'] = geom.get('y')
    rows_equip.append(attr)

df_equip = pd.DataFrame(rows_equip)
gdf_equip = gpd.GeoDataFrame(
    df_equip,
    geometry=gpd.points_from_xy(df_equip['x'], df_equip['y']),
    crs='EPSG:25830'
).to_crs('EPSG:4326')

gdf_equip_barrio = gpd.sjoin(
    gdf_equip[['geometry']],
    gdf_barrios[['coddistbar', 'geometry']],
    how='left', predicate='within'
)

df_equip_por_barrio = gdf_equip_barrio.groupby('coddistbar').size().reset_index(name='n_equipamiento_municipal') 
print(f"Equipamientos municipales: {len(gdf_equip)}")

Equipamientos municipales: 2000


## 9. Dataset: Aparcamientos por barrios y distritos

In [15]:
df_aparca_barrios   = pd.read_csv(DATA_DIR / "aparcamientos_barrios.csv",   encoding='utf-8-sig', sep=None, engine='python')
df_aparca_distritos = pd.read_csv(DATA_DIR / "aparcamientos_distritos.csv", encoding='utf-8-sig', sep=None, engine='python')

print("Aparcamientos barrios:", df_aparca_barrios.shape)
print(df_aparca_barrios.columns.tolist())
print("\nAparcamientos distritos:", df_aparca_distritos.shape)
print(df_aparca_distritos.columns.tolist())

Aparcamientos barrios: (88, 18)
['DISTRICTE', 'BARRI', 'HABITANTS', 'HABITANTS 20-70', 'TOTAL', 'LLIURES', 'ORA', 'GUALS', 'PÀRQUINGS', 'SOLARS', 'ALTRES', 'TURISMES', 'PLACES/HABITANTS', 'PLACES/HABITANTS 20-70', 'PLACES/TURISMES', 'HABITANTS/TURISMES', 'HABITANTS 20-70/TURISMES', 'HABITANTS/HABITANTS 20-70']

Aparcamientos distritos: (19, 18)
['DISTRICTE', 'BARRI', 'HABITANTS', 'HABITANTS 20-70', 'TOTAL', 'LLIURES', 'ORA', 'GUALS', 'PÀRQUINGS', 'SOLARS', 'ALTRES', 'TURISMES', 'PLACES/HABITANTS', 'PLACES/HABITANTS 20-70', 'PLACES/TURISMES', 'HABITANTS/TURISMES', 'HABITANTS 20-70/TURISMES', 'HABITANTS/HABITANTS 20-70']


In [16]:
COL_BARRIO_APARCA    = 'BARRI'
COL_POBLACION_APARCA = 'HABITANTS'

df_aparca_barrios['barrio_key'] = df_aparca_barrios[COL_BARRIO_APARCA].apply(normalizar)
# FIX: alinear el alias 'mont-olivet' con la clave geométrica 'montolivet'
# para que Montolivet reciba su población y no quede como NaN
df_aparca_barrios['barrio_key'] = df_aparca_barrios['barrio_key'].replace('mont-olivet', 'montolivet')

df_poblacion = df_aparca_barrios[['barrio_key', COL_POBLACION_APARCA]].rename(
    columns={COL_POBLACION_APARCA: 'poblacion'}
)

# FIX: tabla barrio → distrito para el join demográfico
# aparcamientos_barrios tiene BARRI y DISTRICTE, que es exactamente lo que necesitamos
df_barrio_distrito = (
    df_aparca_barrios[['barrio_key', 'DISTRICTE']]
    .copy()
    .assign(districte_key=lambda x: x['DISTRICTE'].apply(normalizar))
    [['barrio_key', 'districte_key']]
    .drop_duplicates()
)

print(df_poblacion.head())
print(f"\nMapping barrio→distrito: {len(df_barrio_distrito)} filas")


  barrio_key  poblacion
0     la seu     3013.0
1   la xerea     3908.0
2   el carme     6343.0
3   el pilar     4624.0
4  el mercat     3558.0

Mapping barrio→distrito: 88 filas


## 9bis. Dataset: Índice turístico por barrio
Índice compuesto de presión turística (oferta de alojamiento Airbnb/VUT, actividad de reservas y presión reciente), ya calculado y normalizado a nivel de barrio. Es la fuente principal para el **Perfil C (Gestión de presión turística)**.

In [17]:
# Cargar el índice turístico por barrio (solo nos quedamos con indice_turistico)
df_turismo = pd.read_csv(DATA_DIR / "indice_turistico_por_barrio.csv", encoding='utf-8-sig', sep=None, engine='python')

# FIX: recalculamos barrio_key con la misma función normalizar() del pipeline,
# ya que la barrio_key del CSV usa un criterio distinto (sin separador de guiones,
# p.ej. 'CABANYALCANYAMELAR' en vez de 'cabanyal-canyamelar')
df_turismo['barrio_key'] = df_turismo['barrio'].apply(normalizar)

df_turismo_join = df_turismo[['barrio_key', 'indice_turistico']].copy()

match = set(df_turismo_join['barrio_key']) & set(gdf_barrios['barrio_key'])
print(f"Coinciden por nombre: {len(match)} / {len(df_turismo_join)} turismo, {len(gdf_barrios)} barrios")
faltan = set(gdf_barrios['barrio_key']) - set(df_turismo_join['barrio_key'])
if faltan:
    print(f"⚠ Barrios sin dato de índice turístico: {sorted(faltan)}")

Coinciden por nombre: 88 / 88 turismo, 88 barrios


## 12. Construcción del GeoDataFrame unificado

Aquí hacemos el **JOIN MAESTRO**: partimos de la geometría de barrios y vamos pegando todas las tablas anteriores.

**Nota sobre granularidades:**
- Datos a nivel de **barrio** (88 filas): locales, EMT, metro, equipamiento, vulnerabilidad, población.
- Datos a nivel de **distrito** (19 valores): demografía por edad. Los barrios del mismo distrito comparten estos valores; es la mejor granularidad disponible y perfectamente válido para el modelo.

In [18]:
# Columnas de vulnerabilidad que ya están en gdf_barrios desde la celda 3
# FIX: excluimos Codbar, Nombre, Distrito → son duplicados de codbarrio/nombre/coddistrit
#      y generan 18 NaN (pedanías) que no aportan información nueva al modelo.
vuln_cols = [c for c in ["Ind_Equip", "Vul_Equip", "Ind_Dem", "Vul_Dem",
                          "Ind_Econom", "Vul_Econom", "Ind_Global", "Vul_Global"]
             if c in gdf_barrios.columns]

base_cols = ["coddistbar", "codbarrio", "coddistrit", "nombre", "barrio_key", "geometry"] + vuln_cols
gdf_final = gdf_barrios[base_cols].copy()

gdf_final = gdf_final.merge(df_barrio_distrito, on="barrio_key", how="left")
gdf_final = gdf_final.merge(df_locales_por_barrio, on="coddistbar", how="left")
gdf_final = gdf_final.merge(df_emt_por_barrio,     on="coddistbar", how="left")
gdf_final = gdf_final.merge(df_tren_por_barrio,    on="coddistbar", how="left")
gdf_final = gdf_final.merge(df_equip_por_barrio,   on="coddistbar", how="left")
gdf_final = gdf_final.merge(df_poblacion,           on="barrio_key", how="left")
gdf_final = gdf_final.merge(df_turismo_join,        on="barrio_key", how="left")
gdf_final = gdf_final.merge(df_demo_pivot,          on="districte_key", how="left")
print("GeoDataFrame final:")
print(f"  Filas:    {len(gdf_final)} barrios  ← debe ser 88")
print(f"  Columnas: {gdf_final.shape[1]}")
print(f"  CRS:      {gdf_final.crs}")
gdf_final.head(3)


GeoDataFrame final:
  Filas:    88 barrios  ← debe ser 88
  Columnas: 60
  CRS:      EPSG:4326


,coddistbar,codbarrio,coddistrit,nombre,barrio_key,geometry,Ind_Equip,Vul_Equip,Ind_Dem,Vul_Dem,...,demo_edad_60___64,demo_edad_65___69,demo_edad_70___74,demo_edad_75___79,demo_edad_80___84,demo_edad_85___89,demo_edad_90+,demo_pct_joven,demo_pct_mayor,demo_total_distrito
0,171,1,17,BENIFARAIG,benifaraig,"POLYGON ((-0.37639 39.52831, -0.37667 39.52737...",NaN,NaN,NaN,NaN,...,409.0,368.0,326.0,239.0,207.0,145.0,90.0,24.81,20.94,6566.0
1,161,1,16,BENICALAP,benicalap,"POLYGON ((-0.38139 39.49868, -0.38120 39.49693...",4.6,Vulnerabilidad Baja,3.1,Vulnerabilidad Baja,...,2484.0,2361.0,2177.0,1563.0,1186.0,711.0,385.0,25.13,17.95,46699.0
2,152,2,15,TORREFIEL,torrefiel,"POLYGON ((-0.37122 39.49839, -0.37180 39.49703...",4.2,Vulnerabilidad Baja,2.9,Vulnerabilidad Baja,...,2894.0,2586.0,2323.0,1795.0,1477.0,936.0,382.0,25.96,17.73,53570.0


## 13. Limpieza y valores nulos

In [19]:
# ── Columnas de conteo (n_*): NaN → 0 (barrios sin ningún elemento registrado) ──
cols_conteo = [c for c in gdf_final.columns if c.startswith('n_')]
gdf_final[cols_conteo] = gdf_final[cols_conteo].fillna(0)

# ── Columnas de vulnerabilidad: imputar + flag de dato ausente ────────────────
# Los 18 barrios sin datos son pedanías (distritos 17-19) no cubiertas por el
# dataset municipal. Imputar 0 sería incorrecto ("sin dato" ≠ "sin vulnerabilidad").
# Estrategia por tipo:
#   - Ind_* (float): imputar con la mediana de los barrios que sí tienen dato.
#   - Vul_* (string categórico): imputar con la moda (categoría más frecuente).
vuln_cols_all = [c for c in ['Ind_Equip', 'Vul_Equip', 'Ind_Dem', 'Vul_Dem',
                              'Ind_Econom', 'Vul_Econom', 'Ind_Global', 'Vul_Global']
                 if c in gdf_final.columns]

if vuln_cols_all:
    # Flag ANTES de imputar (1 = el barrio no tiene dato de vulnerabilidad)
    gdf_final['sin_dato_vulnerabilidad'] = gdf_final[vuln_cols_all[0]].isna().astype(int)
    n_sin_dato = gdf_final['sin_dato_vulnerabilidad'].sum()
    print(f"Barrios sin datos de vulnerabilidad (pedanías 17-19): {n_sin_dato}")

    vuln_cols_num = [c for c in vuln_cols_all if gdf_final[c].dtype in ['float64', 'int64']]
    vuln_cols_cat = [c for c in vuln_cols_all if gdf_final[c].dtype not in ['float64', 'int64']]

    # Imputar columnas numéricas (Ind_*) con la mediana
    for col in vuln_cols_num:
        mediana = gdf_final[col].median()
        gdf_final[col] = gdf_final[col].fillna(mediana)
    if vuln_cols_num:
        print(f"  Ind_* imputadas con mediana: {vuln_cols_num}")

    # Imputar columnas categóricas (Vul_*) con la moda
    for col in vuln_cols_cat:
        moda = gdf_final[col].mode()
        if len(moda) > 0:
            gdf_final[col] = gdf_final[col].fillna(moda[0])
    if vuln_cols_cat:
        print(f"  Vul_* imputadas con moda:   {vuln_cols_cat}")
else:
    print("Aviso: no se encontraron columnas de vulnerabilidad en gdf_final")


# ── Columna del índice turístico ───────────────────────────────────────────
# indice_turistico es un score relativo (0-1) calculado a partir de todo el resto
# de barrios; no tiene sentido imputarlo a 0 cuando no hay dato, así que se deja
# como NaN y se marca con un flag, igual que con vulnerabilidad.
if 'indice_turistico' in gdf_final.columns:
    gdf_final['sin_dato_turistico'] = gdf_final['indice_turistico'].isna().astype(int)
    n_sin_turismo = gdf_final['sin_dato_turistico'].sum()
    print(f"Barrios sin dato de índice turístico: {n_sin_turismo}")

# ── Resumen final de nulos ────────────────────────────────────────────────────
nulos = gdf_final.isnull().sum()
nulos_restantes = nulos[nulos > 0]
print("\nColumnas con nulos restantes:")
print(nulos_restantes if len(nulos_restantes) > 0 else "  (ninguna — datos completos)")


Barrios sin datos de vulnerabilidad (pedanías 17-19): 18
  Ind_* imputadas con mediana: ['Ind_Equip', 'Ind_Dem', 'Ind_Econom', 'Ind_Global']
  Vul_* imputadas con moda:   ['Vul_Equip', 'Vul_Dem', 'Vul_Econom', 'Vul_Global']
Barrios sin dato de índice turístico: 0

Columnas con nulos restantes:
  (ninguna — datos completos)


In [20]:
# Verificación: el replace mont-olivet→montolivet se aplica ya en la celda de
# carga de aparcamientos (sección 9). Aquí comprobamos que no queden barrios
# con nombre desalineado en df_aparca_barrios.
barrios_no_alineados = set(df_aparca_barrios['barrio_key']) - set(gdf_barrios['barrio_key'])
if barrios_no_alineados:
    print(f"⚠ Barrios en aparcamientos sin match en geometría: {sorted(barrios_no_alineados)}")
else:
    print("✓ Todos los barrios de aparcamientos tienen match en la geometría base.")


✓ Todos los barrios de aparcamientos tienen match en la geometría base.


## 14. Feature Engineering: variables derivadas para el modelo

In [21]:
# Densidad de locales por habitante (KPI principal)
gdf_final['densidad_locales_hab'] = np.where(
    gdf_final['poblacion'] > 0,
    gdf_final['n_locales_total'] / gdf_final['poblacion'],
    np.nan
)

# Índice de accesibilidad en transporte público (metro pondera x2)
gdf_final['accesibilidad_tp'] = (
    gdf_final['n_paradas_emt'].fillna(0) +
    gdf_final['n_paradas_metro'].fillna(0) * 2
)

# Área del barrio en km² (para normalizar)
gdf_utm = gdf_final.to_crs(epsg=25830)
gdf_final['area_km2'] = gdf_utm.geometry.area / 1e6

# Densidad de locales por km²
gdf_final['densidad_locales_km2'] = gdf_final['n_locales_total'] / gdf_final['area_km2']

print("Variables derivadas creadas: densidad_locales_hab, accesibilidad_tp, area_km2, densidad_locales_km2")
gdf_final[['nombre', 'n_locales_total', 'poblacion', 'densidad_locales_hab', 'accesibilidad_tp']].sort_values('densidad_locales_hab').head(10)

Variables derivadas creadas: densidad_locales_hab, accesibilidad_tp, area_km2, densidad_locales_km2


,nombre,n_locales_total,poblacion,densidad_locales_hab,accesibilidad_tp
79,EL SALER,0.0,1836.0,0.000000,11.0
77,EL PERELLONET,0.0,1378.0,0.000000,17.0
78,EL PALMAR,0.0,755.0,0.000000,8.0
66,ELS ORRIOLS,58.0,16423.0,0.003532,23.0
72,FAITANAR,5.0,1295.0,0.003861,3.0
3,TORMOS,35.0,8628.0,0.004057,6.0
82,CARPESA,5.0,1186.0,0.004216,6.0
65,CIUTAT FALLERA,28.0,5791.0,0.004835,6.0
35,TRES FORQUES,47.0,9011.0,0.005216,15.0
46,NA ROVELLA,42.0,7815.0,0.005374,15.0


## 15. Etiqueta target: Clasificación de Oferta Insuficiente

Definimos el umbral crítico para el **Tema 1 (Model Evaluation)**. Los barrios por debajo del percentil 25 en densidad de locales se clasifican como `oferta_insuficiente = 1`.

In [22]:
PERCENTIL_UMBRAL = 25

# quantile() ignora NaN por defecto → el umbral se calcula solo con barrios válidos. OK.
umbral = gdf_final['densidad_locales_hab'].quantile(PERCENTIL_UMBRAL / 100)

# Los barrios sin densidad calculable (poblacion == 0 o NaN) reciben pd.NA,
# NO se etiquetan como clase 0 (no-insuficiente) de forma incorrecta.
# Tras el fix de mont-olivet→montolivet en la celda 27, solo Rafalell-Vistabella
# (población real = 0) quedará excluido; Montolivet tendrá su valor correcto.
gdf_final['oferta_insuficiente'] = pd.NA
mask_valido = gdf_final['densidad_locales_hab'].notna()
gdf_final.loc[mask_valido, 'oferta_insuficiente'] = (
    gdf_final.loc[mask_valido, 'densidad_locales_hab'] < umbral
).astype(int)

print(f"Umbral (percentil {PERCENTIL_UMBRAL}): {umbral:.5f} locales/habitante")
print(f"Barrios con oferta insuficiente (clase 1): {(gdf_final['oferta_insuficiente']==1).sum()} / {mask_valido.sum()} válidos")
print(f"Barrios excluidos por densidad no calculable (pd.NA): {(~mask_valido).sum()}")
print("  → Solo Rafalell-Vistabella debería quedar excluida (población real = 0)")
print()
print(gdf_final['oferta_insuficiente'].value_counts(dropna=False))


Umbral (percentil 25): 0.00823 locales/habitante
Barrios con oferta insuficiente (clase 1): 22 / 87 válidos
Barrios excluidos por densidad no calculable (pd.NA): 1
  → Solo Rafalell-Vistabella debería quedar excluida (población real = 0)

oferta_insuficiente
0       65
1       22
<NA>     1
Name: count, dtype: int64


## 16. Exportación de resultados

In [23]:
!pip install "numpy<2"



In [24]:
OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# FIX: exportar GeoJSON sin claves auxiliares de join (barrio_key, districte_key)
# que son artefactos internos del pipeline, no features del modelo ni del mapa.
cols_drop_geo = ["barrio_key", "districte_key"]
gdf_export = gdf_final.drop(columns=[c for c in cols_drop_geo if c in gdf_final.columns])
gdf_export.to_file(OUTPUT_DIR / "geomarket_vlc_barrios.geojson", driver="GeoJSON")

# CSV para modelado: sin geometría ni claves auxiliares
# FIX: excluir la fila de Rafalell-Vistabella (poblacion=0, target=NaN)
#      para que el CSV de entrenamiento no tenga filas sin etiquetar.
cols_drop_csv = ["geometry", "barrio_key", "districte_key"]
df_modelo = (
    gdf_final
    .drop(columns=[c for c in cols_drop_csv if c in gdf_final.columns])
    .dropna(subset=["oferta_insuficiente"])   # elimina Rafalell-Vistabella (target NaN)
)
df_modelo.to_csv(OUTPUT_DIR / "geomarket_vlc_features.csv", index=False, encoding="utf-8-sig")

print("Archivos guardados en ./output/:")
print(f"  - geomarket_vlc_barrios.geojson  → {len(gdf_export)} barrios  (mapa coroplético)")
print(f"  - geomarket_vlc_features.csv     → {len(df_modelo)} barrios etiquetados  (sklearn)")


Archivos guardados en ./output/:
  - geomarket_vlc_barrios.geojson  → 88 barrios  (mapa coroplético)
  - geomarket_vlc_features.csv     → 87 barrios etiquetados  (sklearn)


## 17. Visualización rápida del mapa unificado

In [25]:
!pip install folium

In [26]:
import folium
from folium.features import GeoJsonTooltip
import json

# Copia limpia para folium: coddistbar como int estándar (no Int64 nullable)
gdf_plot = gdf_final.copy()
gdf_plot["coddistbar"] = gdf_plot["coddistbar"].fillna(0).astype(int)
gdf_plot["target_plot"] = gdf_plot["oferta_insuficiente"].fillna(-1).astype(int)
gdf_plot["densidad_fmt"] = gdf_plot["densidad_locales_hab"].map(
    lambda x: f"{x:.5f}" if pd.notna(x) else "N/D"
)
geo_data = json.loads(gdf_plot.to_json())

# Mapa base
m = folium.Map(location=[39.4699, -0.3763], zoom_start=12, tiles="CartoDB positron")

# MAPA 1: Escala Lineal
folium.Choropleth(
    geo_data=geo_data,
    data=gdf_plot,
    columns=["coddistbar", "densidad_locales_hab"],
    key_on="feature.properties.coddistbar",
    fill_color="RdYlGn",
    fill_opacity=0.75,
    line_opacity=0.3,
    legend_name="Densidad Locales/Hab (Lineal)",
    nan_fill_color="lightgrey",
    name="1. Escala Lineal (Original)",
    show=True
).add_to(m)

# MAPA 2: Escala por Percentiles
valores_validos = gdf_plot["densidad_locales_hab"].dropna()
cortes_percentiles = list(valores_validos.quantile([0, 0.25, 0.50, 0.75, 0.90, 1.0]))
cortes_percentiles[0] = valores_validos.min() - 0.00001
cortes_percentiles[-1] = valores_validos.max() + 0.00001

folium.Choropleth(
    geo_data=geo_data,
    data=gdf_plot,
    columns=["coddistbar", "densidad_locales_hab"],
    key_on="feature.properties.coddistbar",
    fill_color="RdYlGn",
    fill_opacity=0.75,
    line_opacity=0.3,
    bins=cortes_percentiles,
    legend_name="Densidad Locales/Hab (Percentiles)",
    nan_fill_color="lightgrey",
    name="2. Escala por Percentiles",
    show=False
).add_to(m)

# MAPA 3: Target Binario
folium.Choropleth(
    geo_data=geo_data,
    data=gdf_plot,
    columns=["coddistbar", "target_plot"],
    key_on="feature.properties.coddistbar",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    bins=[-1.1, -0.1, 0.9, 1.1],
    legend_name="Clasificación Target (0: Suficiente, 1: Insuficiente)",
    nan_fill_color="lightgrey",
    name="3. Target Binario (Modelo)",
    show=False
).add_to(m)

# Tooltip interactivo
folium.GeoJson(
    geo_data,
    tooltip=GeoJsonTooltip(
        fields=["nombre", "n_locales_total", "poblacion", "densidad_fmt", "oferta_insuficiente"],
        aliases=["Barrio:", "Total Locales:", "Población:", "Densidad (Loc/Hab):", "Oferta Insuficiente (Target):"],
        sticky=True
    ),
    style_function=lambda x: {"fillOpacity": 0, "color": "#444", "weight": 0.5}
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m

## 18. Resumen del GeoDataFrame final

In [27]:
print("=" * 55)
print("RESUMEN: GeoDataFrame unificado GeoMarket-VLC")
print("=" * 55)
print(f"Barrios totales:     {len(gdf_final)}")
print(f"Barrios modelables:  {len(df_modelo)}  (Rafalell-Vistabella excluida: pob=0, target=NaN)")
print(f"Features totales:    {gdf_final.shape[1] - 1} (sin geometría)")
n_clase1 = (gdf_final["oferta_insuficiente"] == 1).sum()
print(f"Variable target:     oferta_insuficiente  ({n_clase1} barrios en clase 1)")
print("Fuentes integradas:")
print("  ✓ locales_valencia.json        → n_locales_total, n_locales_<tipo>")
print("  ✓ emt_paradas.json             → n_paradas_emt")
print("  ✓ fgv_bocas.json               → n_paradas_metro")
print("  ✓ equipament_municipal.json    → n_equipamiento_municipal")
print("  ✓ vulnerabilidad-por-barrios   → índices socioeconómicos")
print("  ✓ aparcamientos_barrios        → población")
print("  ✓ demografia_distritos         → demo_pct_joven, demo_pct_mayor (por distrito)")
print("  ✓ indice_turistico_por_barrio  → índice de presión turística (Airbnb/VUT)")
print("Output listo para:")
print("  → Modelado (Temas 1, 2, 3)     [geomarket_vlc_features.csv]  ← 87 barrios etiquetados")
print("  → Mapa Streamlit/Folium        [geomarket_vlc_barrios.geojson]  ← 88 barrios")


RESUMEN: GeoDataFrame unificado GeoMarket-VLC
Barrios totales:     88
Barrios modelables:  87  (Rafalell-Vistabella excluida: pob=0, target=NaN)
Features totales:    66 (sin geometría)
Variable target:     oferta_insuficiente  (22 barrios en clase 1)
Fuentes integradas:
  ✓ locales_valencia.json        → n_locales_total, n_locales_<tipo>
  ✓ emt_paradas.json             → n_paradas_emt
  ✓ fgv_bocas.json               → n_paradas_metro
  ✓ equipament_municipal.json    → n_equipamiento_municipal
  ✓ vulnerabilidad-por-barrios   → índices socioeconómicos
  ✓ aparcamientos_barrios        → población
  ✓ demografia_distritos         → demo_pct_joven, demo_pct_mayor (por distrito)
  ✓ indice_turistico_por_barrio  → índice de presión turística (Airbnb/VUT)
Output listo para:
  → Modelado (Temas 1, 2, 3)     [geomarket_vlc_features.csv]  ← 87 barrios etiquetados
  → Mapa Streamlit/Folium        [geomarket_vlc_barrios.geojson]  ← 88 barrios
